In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

categories = ['Dairy', 'Bakery', 'Fresh produce', 'Pantry', 'Meat', 'Household']
promo_rate = {'Dairy': 0.12, 'Bakery': 0.12, 'Fresh produce': 0.12,
              'Pantry': 0.15, 'Meat': 0.15, 'Household': 0.25}

max_discount = {'Household': 0.40}
store_uplift = {'Metro': 1.00, 'Regional': 1.04}

n_products, n_weeks = 50, 52
weeks = pd.date_range(start='2025-01-06', periods=n_weeks, freq='W-MON')

products = pd.DataFrame({
    'product_id': np.arange(1, n_products + 1),
    'category': rng.choice(categories, size=n_products),
    'base_price': rng.lognormal(mean=1.5, sigma=0.5, size=n_products).round(2),
})
products.head()

,product_id,category,base_price
0,1,Dairy,3.76
1,2,Meat,5.85
2,3,Pantry,5.38
3,4,Fresh produce,5.51
4,5,Fresh produce,5.56


In [3]:
rows = []
for p in products.itertuples():
    regular = p.base_price
    for week in weeks:
        regular *= 1 + rng.normal(0, 0.005)  # small weekly drift
        on_promo = rng.random() < promo_rate[p.category]
        depth = rng.uniform(0.10, max_discount.get(p.category, 0.30)) if on_promo else 0.0
        for store, uplift in store_uplift.items():
            store_regular = regular * uplift * (1 + rng.normal(0, 0.02))
            rows.append({
                'product_id': p.product_id,
                'category': p.category,
                'week': week,
                'store_group': store,
                'regular_price': round(store_regular, 2),
                'shelf_price': round(store_regular * (1 - depth), 2),
                'on_promo': on_promo,
                'discount_depth': round(depth, 3),
            })
        if on_promo and rng.random() < 0.3:
            regular *= 1.03  # small creep after some promotions

df = pd.DataFrame(rows)

In [4]:
df.to_csv('data/simulated_prices.csv', index=False)
print(df.shape)
df.head()

(5200, 8)


,product_id,category,week,store_group,regular_price,shelf_price,on_promo,discount_depth
0,1,Dairy,2025-01-06,Metro,3.78,3.78,False,0.0
1,1,Dairy,2025-01-06,Regional,3.97,3.97,False,0.0
2,1,Dairy,2025-01-13,Metro,3.72,3.72,False,0.0
3,1,Dairy,2025-01-13,Regional,3.89,3.89,False,0.0
4,1,Dairy,2025-01-20,Metro,3.80,3.80,False,0.0


In [5]:
df[(df['category'] == 'Dairy') & (df['on_promo'])].head()

,product_id,category,week,store_group,regular_price,shelf_price,on_promo,discount_depth
10,1,Dairy,2025-02-10,Metro,3.76,2.66,True,0.292
11,1,Dairy,2025-02-10,Regional,3.81,2.70,True,0.292
14,1,Dairy,2025-02-24,Metro,3.93,2.83,True,0.281
15,1,Dairy,2025-02-24,Regional,3.88,2.79,True,0.281
20,1,Dairy,2025-03-17,Metro,3.87,3.16,True,0.183


In [6]:
df.groupby('category')['shelf_price'].mean().round(2)

category
Bakery           4.62
Dairy            4.90
Fresh produce    5.19
Household        5.67
Meat             4.58
Pantry           5.42
Name: shelf_price, dtype: float64

In [7]:
promo_counts = (df[df['store_group'] == 'Metro']
                .groupby('product_id')['on_promo'].sum()
                .rename('promo_weeks').reset_index())
products.merge(promo_counts, on='product_id').head()

,product_id,category,base_price,promo_weeks
0,1,Dairy,3.76,10
1,2,Meat,5.85,16
2,3,Pantry,5.38,9
3,4,Fresh produce,5.51,5
4,5,Fresh produce,5.56,4


In [8]:
promo_counts = (df[df['store_group'] == 'Metro']
                .groupby('product_id')['on_promo'].sum()
                .rename('promo_weeks').reset_index())
products.merge(promo_counts, on='product_id').head()

,product_id,category,base_price,promo_weeks
0,1,Dairy,3.76,10
1,2,Meat,5.85,16
2,3,Pantry,5.38,9
3,4,Fresh produce,5.51,5
4,5,Fresh produce,5.56,4
